# Society Feature Ablation - Is `society` Earning Its Keep?

**Diagnostic experiment - NOT part of the training pipeline.**

> ⚠️ **Historical record (run 2026-07-16).** Every number below was measured under the setup in
> place at that time: a two-way 80/20 train/test split, model family selected on test, and
> duplicate feature-rows still present. Those flaws were fixed afterwards, so the numbers here are
> **not** comparable to the current headline metric — the deployed model today is **LightGBM at
> 11.39% MAPE** on a properly held-out test set (60/20/20 split, family chosen on validation,
> duplicates removed).
>
> The numbers are deliberately left as they were. Both sides of the comparison were measured the
> same way, so **the conclusion still holds** — and rewriting them would misrepresent what was
> actually run. This notebook loads the *archived* 25-feature society model, so it stays runnable.

## The question

The model of the day was trained with `society` as a feature and scored **10.65% MAPE** on the test
set. But that test used each listing's **true** society - information production never has. A user
can't reliably name their society (there are 660 of them), so the app **guessed** it: the most
common society in the sector they picked.

So the honest question is not *"how good is the model on the test set?"* but
**"how good is the model with the information it will actually receive?"**

## The method

A controlled experiment - same trained model, same test rows, same metric. **Only one thing changes:**
the `society` column.

| Scenario | Society value | What it represents |
|---|---|---|
| A | true society | the offline test metric |
| B | guessed from sector | **what production actually did** |

Scenario B is then compared against a separately retrained **no-society model (11.18% MAPE)**.

## 1. Reproduce the test set, with society intact

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, r2_score

# Find the project root from wherever this notebook is opened.
ROOT = Path.cwd()
while not (ROOT / 'artifacts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# The ARCHIVED 25-feature society model this experiment is about. We deliberately
# do NOT load artifacts/best_model.joblib: that now holds the 24-feature model,
# which has never heard of society and would reject these columns outright.
SOCIETY_MODEL = ROOT / 'artifacts/best_model_20260714_160533.joblib'

# The 25 features that model was trained on (society included).
# NOTE: data/fs/ has since been regenerated WITHOUT society, so we step back one
# stage to data/pp/ which still holds every column.
FEATURES_WITH_SOCIETY = [
    'area', 'dist_to_golf_road', 'total_floor', 'bathroom', 'property_type',
    'dist_to_cyber_city', 'society', 'covered_parking', 'dist_to_manesar', 'sector',
    'dist_to_airport', 'bedRoom', 'furnishing', 'balcony', 'age_possession_category',
    'facing', 'has_ac', 'total_parking', 'open_parking', 'has_power_backup',
    'is_corner', 'floornum_category', 'ov_main_road', 'has_pool', 'ov_others',
]
TARGET = 'price_in_cr'

df = pd.read_csv(ROOT / 'data/pp/preprocessed_properties.csv')[FEATURES_WITH_SOCIETY + [TARGET]]

# Reproduce the two-way split that was in use when this experiment was run.
# (The pipeline now uses a 60/20/20 three-way split — see the note at the top.)
X, y = df.drop(columns=[TARGET]), df[TARGET]
y_log = np.log1p(y)
bins = pd.qcut(y, q=5, labels=False)
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, stratify=bins, test_size=0.2, random_state=42)

print('Test rows:', len(X_test))

## 2. Score the deployed model twice - true vs guessed society

In [ ]:
bundle = joblib.load(SOCIETY_MODEL)          # the archived 25-feature model
pipe = bundle['pipeline']
print('Model under test:', bundle['model_name'], '| advertised MAPE:', bundle['test_mape_percent'], '%')

y_true = np.expm1(y_test_log)   # predictions are log-price; convert back to Crores

# A) TRUE society - the flattering number (information production never has)
pred_true = np.expm1(pipe.predict(X_test))
mape_true = mean_absolute_percentage_error(y_true, pred_true) * 100

# B) GUESSED society - what the app did at the time: sector's most common society.
# sector_reference.csv has since lost its society column, so the guess is rebuilt
# here from the data exactly as that lookup table was originally built.
modal_society = (df.loc[df['society'] != 'other']
                   .groupby('sector')['society']
                   .agg(lambda s: s.mode().iloc[0]))
X_test_guessed = X_test.copy()
X_test_guessed['society'] = X_test_guessed['sector'].map(modal_society).fillna('other')
pred_guessed = np.expm1(pipe.predict(X_test_guessed))
mape_guessed = mean_absolute_percentage_error(y_true, pred_guessed) * 100

# How often is the guess even correct?
match_rate = (X_test['society'] == X_test_guessed['society']).mean() * 100

print()
print(f'A) TRUE society    : {mape_true:6.2f}%   <- the flattering number')
print(f'B) GUESSED society : {mape_guessed:6.2f}%   <- what production really got')
print(f'C) No-society model:  11.18%   <- separate retrain, same 2-way-split setup')
print()
print(f'Guess matches true society : {match_rate:.1f}% of the time')
print(f'True societies in test set : {X_test["society"].nunique()}')
print(f'Distinct guesses possible  : {X_test_guessed["society"].nunique()}')

## 3. Log the finding to MLflow

This run **trains nothing** - it records a *measurement* of an existing model under production
conditions. That's a perfectly normal use of experiment tracking.

Note the deliberate care with naming/tags/params: the earlier runs were all called `LightGBM`
with nothing recording whether society was included, which made the MLflow UI unreadable.
A run should explain itself without you having to remember anything.

In [ ]:
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')   # explicit path: a notebook's cwd is not the project root

import mlflow

mlflow.set_experiment('propnavigator-model-building')   # same folder as the other runs

with mlflow.start_run(run_name='LightGBM-society-GUESSED-production-sim'):
    mlflow.set_tag('society', 'with_society')
    mlflow.set_tag('evaluation', 'production_simulation')
    mlflow.set_tag('note', 'Deployed model re-scored with society guessed from sector, as production does')

    mlflow.log_param('model_type', 'LightGBM')
    mlflow.log_param('n_features', 25)
    mlflow.log_param('society_source', 'guessed_from_sector_modal')

    mlflow.log_metric('test_mape', round(mape_guessed, 2))
    mlflow.log_metric('test_r2', round(r2_score(y_true, pred_guessed), 4))
    mlflow.log_metric('society_guess_accuracy_pct', round(match_rate, 1))

print('Logged to MLflow.')

## 4. Conclusion

The real choice was never *10.65% vs 11.18%*. It was **14.29% vs 11.18%**.

| Scenario | MAPE | Reality |
|---|---|---|
| Society model, true society | 10.65% | a fantasy - needs info we never have |
| Society model, guessed society | **14.29%** | what the app delivered at the time |
| No-society model | **11.18%** | the honest alternative |

Dropping `society` **improved production accuracy by ~3.1 points**.

**Why "a wrong value beats no value" is a myth:** a model can't tell a fact from a guess. With no
society feature the model simply leans on area/sector/bedrooms - neutral. With a *wrong* society it
acts on a confident lie, because it learned during training that the column is trustworthy.

**The lesson:** never train on a feature you can't obtain honestly at serving time. A feature that
only exists in training is a liability, not an asset. This is textbook **train/serve skew**.

---

### What happened next

All three numbers above share a flaw that was found later: the model family was selected on the
**test** set, and duplicate feature-rows were still present (252 test rows had an identical twin in
train, which the model could memorise). Both were fixed:

- 60/20/20 **train/validation/test** split — the winning family is now chosen on validation, and
  test is scored exactly once
- 831 duplicate feature-rows dropped before splitting

Re-measured under those stricter rules, the deployed model scores **11.39% MAPE (R² 0.91)**. The
~0.2-point gap against the old 11.18% is the inflation that the old setup was hiding — small, but
now quantified rather than assumed.

None of this changes the society conclusion: both arms of that comparison were measured identically,
so the 3-point gap between them stands.